In [1]:

%pip install -q ultralytics mlflow datasets huggingface_hub pyyaml


In [2]:

from google.colab import drive
from pathlib import Path
import shutil
import torch
import zipfile

drive.mount("/content/drive")

ROOT = Path("/content/roadvision")
DRIVE_ROOT = Path("/content/drive/MyDrive/RoadVision")

DATASET_DIR = ROOT / "datasets/rdd2022"
V2_DIR = ROOT / "datasets/rdd2022_v2"

archive = DRIVE_ROOT / "datasets/rdd2022_subset_v1.zip"

assert archive.exists(), "Missing v1 dataset backup!"

if not (DATASET_DIR / "images/train").exists():
    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive) as z:
        z.extractall(DATASET_DIR)

print("GPU available:", torch.cuda.is_available())

for split in ["train", "val", "test"]:
    p = DATASET_DIR / "images" / split
    images = [x for x in p.iterdir()
              if x.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    print(split, len(images))


Mounted at /content/drive
GPU available: True
train 300
val 75
test 75


In [3]:

import shutil

if not V2_DIR.exists():
    shutil.copytree(DATASET_DIR, V2_DIR)

print("Created v2 dataset:", V2_DIR)


Created v2 dataset: /content/roadvision/datasets/rdd2022_v2


In [4]:

import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

existing_hashes = set()

for split in ["train", "val", "test"]:
    image_dir = V2_DIR / "images" / split

    for path in image_dir.iterdir():
        if path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            existing_hashes.add(sha256_file(path))

print("Existing unique image hashes:", len(existing_hashes))


Existing unique image hashes: 450


In [5]:

from datasets import load_dataset
from huggingface_hub import HfApi
from pathlib import PurePosixPath

DATASET_ID = "dronefreak/RDD2022"

stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True
)

stream = stream.shuffle(seed=2026, buffer_size=2000)

# Get more candidates than needed to allow deduplication
candidates = list(stream.take(1200))

print("Candidate metadata rows:", len(candidates))

api = HfApi()
repo_files = api.list_repo_files(
    repo_id=DATASET_ID,
    repo_type="dataset"
)

image_lookup = {}

for path in repo_files:
    p = PurePosixPath(path)

    if "/images/train/" in f"/{path}" and \
       p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        image_lookup.setdefault(p.name, []).append(path)

print("Indexed train image filenames:", len(image_lookup))


README.md:   0%|          | 0.00/11.1k [00:00<?, ?B/s]

Candidate metadata rows: 1200
Indexed train image filenames: 26869


In [6]:

from huggingface_hub import hf_hub_download
from PIL import Image
import shutil

CLASS_NAMES = {
    0: "longitudinal_crack",
    1: "transverse_crack",
    2: "alligator_crack",
    3: "pothole"
}

def to_yolo(objects, width, height):
    lines = []

    for box, cls in zip(
        objects["bbox"],
        objects["categories"]
    ):
        cls = int(cls)
        if cls not in CLASS_NAMES:
            continue

        x, y, w, h = map(float, box)

        x1 = max(0, min(x, width))
        y1 = max(0, min(y, height))
        x2 = max(0, min(x+w, width))
        y2 = max(0, min(y+h, height))

        if x2 <= x1 or y2 <= y1:
            continue

        xc = (x1+x2)/(2*width)
        yc = (y1+y2)/(2*height)
        bw = (x2-x1)/width
        bh = (y2-y1)/height

        lines.append(
            f"{cls} {xc:.6f} {yc:.6f} "
            f"{bw:.6f} {bh:.6f}"
        )

    return lines


img_dir = V2_DIR / "images/train"
label_dir = V2_DIR / "labels/train"

added = 0
TARGET_NEW = 700
failed = []

for row in candidates:
    if added >= TARGET_NEW:
        break

    name = row["file_name"]
    paths = image_lookup.get(name, [])

    if len(paths) != 1:
        failed.append(name)
        continue

    source_path = hf_hub_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        filename=paths[0]
    )

    image_hash = sha256_file(source_path)

    if image_hash in existing_hashes:
        continue

    with Image.open(source_path) as img:
        width, height = img.size

    new_stem = f"train_extra_{added:05d}"
    new_image = img_dir / (new_stem + Path(name).suffix.lower())
    new_label = label_dir / (new_stem + ".txt")

    yolo_labels = to_yolo(
        row["objects"],
        width,
        height
    )

    shutil.copy2(source_path, new_image)
    new_label.write_text(
        "\n".join(yolo_labels),
        encoding="utf-8"
    )

    existing_hashes.add(image_hash)
    added += 1

    if added % 100 == 0:
        print(f"Added {added}/{TARGET_NEW} images")

print("New images:", added)
print("Unresolved paths:", len(failed))

if added != TARGET_NEW:
    print("Not enough candidates. Expand sampling before training.")


data/images/train/shard_000/China_MotorB(…): reconstructing file:   0%|          |  0.00B / 88.9kB            

data/images/train/shard_000/China_MotorB(…): downloading bytes:           |  0.00B            

data/images/train/shard_000/China_Drone_(…): reconstructing file:   0%|          |  0.00B / 69.6kB            

data/images/train/shard_000/China_Drone_(…): downloading bytes:           |  0.00B            

data/images/train/shard_000/China_Drone_(…): reconstructing file:   0%|          |  0.00B / 75.5kB            

data/images/train/shard_000/China_Drone_(…): downloading bytes:           |  0.00B            

... (1,413 outputs left out)

data/images/train/shard_000/China_Drone_(…): downloading bytes:           |  0.00B            

data/images/train/shard_000/China_Drone_(…): reconstructing file:   0%|          |  0.00B / 61.2kB            

data/images/train/shard_000/China_Drone_(…): downloading bytes:           |  0.00B            

Added 700/700 images
New images: 700
Unresolved paths: 0


In [7]:

import yaml

data_config = {
    "path": str(V2_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CLASS_NAMES
}

yaml_v2 = V2_DIR / "data.yaml"

with open(yaml_v2, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

counts = {}

for split in ["train", "val", "test"]:
    image_dir = V2_DIR / "images" / split
    label_dir = V2_DIR / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    for p in images:
        label = label_dir / (p.stem + ".txt")
        assert label.exists(), f"Missing label for {p.name}"

    counts[split] = len(images)

print("Dataset v2:", counts)

assert counts == {
    "train": 1000,
    "val": 75,
    "test": 75
}, "Unexpected dataset sizes!"

# Ensure evaluation images have not changed
for split in ["val", "test"]:
    old_files = sorted(
        (DATASET_DIR / "images" / split).iterdir()
    )

    for original in old_files:
        copied = V2_DIR / "images" / split / original.name
        assert copied.exists()
        assert sha256_file(original) == sha256_file(copied)

print("Validation and test sets unchanged!")


Dataset v2: {'train': 1000, 'val': 75, 'test': 75}
Validation and test sets unchanged!


In [8]:

import mlflow
import os
import shutil

TRACKING_ROOT = Path("/content/roadvision_mlflow")
DB_PATH = TRACKING_ROOT / "mlflow.db"
ARTIFACT_ROOT = TRACKING_ROOT / "artifacts"
BACKUP_ROOT = DRIVE_ROOT / "mlflow_backup"

TRACKING_ROOT.mkdir(parents=True, exist_ok=True)

# Restore only when local DB does not exist
if not DB_PATH.exists():
    if (BACKUP_ROOT / "mlflow.db").exists():
        shutil.copy2(
            BACKUP_ROOT / "mlflow.db",
            DB_PATH
        )

    if (BACKUP_ROOT / "artifacts").exists():
        shutil.copytree(
            BACKUP_ROOT / "artifacts",
            ARTIFACT_ROOT,
            dirs_exist_ok=True
        )

ARTIFACT_ROOT.mkdir(exist_ok=True)

TRACKING_URI = f"sqlite:///{DB_PATH}"

mlflow.set_tracking_uri(TRACKING_URI)

os.environ["MLFLOW_TRACKING_URI"] = TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = "RoadVision-Detection"
os.environ["MLFLOW_RUN"] = "experiment-v2-1000-images"

mlflow.set_experiment("RoadVision-Detection")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", "RoadVision-Detection")


Tracking URI: sqlite:////content/roadvision_mlflow/mlflow.db
Experiment: RoadVision-Detection


In [9]:

from ultralytics import YOLO, settings
import mlflow

settings.update({"mlflow": True})

# Ultralytics manages its MLflow run
assert mlflow.active_run() is None

print("MLflow integration enabled!")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
MLflow integration enabled!


In [10]:

import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"

model_v2 = YOLO("yolo11n.pt")

train_results_v2 = model_v2.train(
    data=str(yaml_v2),
    epochs=20,
    imgsz=640,
    batch=8,
    device=device,
    patience=7,
    seed=42,
    project=str(ROOT / "runs"),
    name="roadvision_yolo11n_v2",
    exist_ok=True,
    plots=True
)

print("Experiment v2 training complete!")


Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/roadvision/datasets/rdd2022_v2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=roadvision_yolo11n_v2, nbs=64, nms=None, 

2026/10/29 00:49:45 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/10/29 00:49:45 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(eced125947cd452185650c4f3a843526) to sqlite:////content/roadvision_mlflow/mlflow.db
MLflow: disable with 'yolo settings mlflow=False'
Using 1000 train, 75 val images for fraction=1.0 at imgsz=640
Using 2 dataloader workers
Logging results to /content/roadvision/runs/roadvision_yolo11n_v2
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/20      1.26G      2.174      4.468      2.013         28        640: 100% ━━━━━━━━━━━━ 125/125 4.8it/s 26.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
                   all         75        139     0.0806      0.111      0.047      0.017

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/20      1.49G      2.143       3.29      1.963         15        640: 100% ━━━━━━━━━━━━ 125/125 6.0it/s 20.7s
                 Class     Images  Instance

In [11]:

best_v2_path = (
    ROOT / "runs" /
    "roadvision_yolo11n_v2" /
    "weights" / "best.pt"
)

assert best_v2_path.exists()

best_v2 = YOLO(str(best_v2_path))

metrics_v2 = best_v2.val(
    data=str(yaml_v2),
    split="val",
    device=device,
    plots=True
)

v2_results = {
    "precision": float(metrics_v2.box.mp),
    "recall": float(metrics_v2.box.mr),
    "map50": float(metrics_v2.box.map50),
    "map50_95": float(metrics_v2.box.map)
}

print(v2_results)


Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,780 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1761.1±683.0 MB/s, size: 66.0 KB)
val: Scanning /content/roadvision/datasets/rdd2022_v2/labels/val.cache... 75 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 75/75 22.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.5it/s 3.3s
                   all         75        139      0.572      0.275      0.319      0.164
    longitudinal_crack         35         65      0.596      0.272      0.338      0.168
      transverse_crack         20         38      0.451      0.237      0.231     0.0828
       alligator_crack         17         23      0.566      0.285      0.359      0.135
               pothole         11         13      0.675      0.308      0.349      0.272
Speed: 7.6ms preprocess, 8.0

In [12]:

from mlflow.tracking import MlflowClient

client = MlflowClient()
exp = client.get_experiment_by_name(
    "RoadVision-Detection"
)

runs = client.search_runs(
    [exp.experiment_id],
    order_by=["attributes.start_time DESC"],
    max_results=100
)

v2_runs = [
    r for r in runs
    if r.data.tags.get("mlflow.runName")
    in {
        "roadvision_yolo11n_v2",
        "experiment-v2-1000-images"
    }
]

assert v2_runs, "V2 MLflow run not found"

v2_run = v2_runs[0]

for key, value in v2_results.items():
    client.log_metric(
        v2_run.info.run_id,
        f"final_val_{key}",
        value
    )

client.set_tag(
    v2_run.info.run_id,
    "dataset_version",
    "rdd2022-subset-v2"
)

client.set_tag(
    v2_run.info.run_id,
    "train_images",
    "1000"
)

print("Updated run:", v2_run.info.run_id)


Updated run: eced125947cd452185650c4f3a843526


In [13]:

import pandas as pd

v1_results = {
    "precision": 0.2214061858247467,
    "recall": 0.1858431614152438,
    "map50": 0.12550362730956546,
    "map50_95": 0.05621294291432901
}

comparison = pd.DataFrame({
    "Metric": list(v1_results.keys()),
    "Baseline v1": list(v1_results.values()),
    "Experiment v2": [
        v2_results[k] for k in v1_results
    ]
})

comparison["Improvement"] = (
    comparison["Experiment v2"] -
    comparison["Baseline v1"]
)

display(comparison)


,Metric,Baseline v1,Experiment v2,Improvement
0,precision,0.221406,0.571912,0.350506
1,recall,0.185843,0.275314,0.089471
2,map50,0.125504,0.319299,0.193795
3,map50_95,0.056213,0.164340,0.108127


In [14]:

import shutil
import json
import hashlib

V2_DRIVE = DRIVE_ROOT / "models/v2"
V2_DRIVE.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_v2_path, V2_DRIVE / "best.pt")
shutil.copy2(yaml_v2, V2_DRIVE / "data.yaml")

metadata_v2 = {
    "model": "YOLO11n",
    "version": "v2",
    "dataset": "RDD2022 subset v2",
    "train_images": 1000,
    "val_images": 75,
    "test_images": 75,
    "metrics": v2_results,
    "mlflow_run_id": v2_run.info.run_id,
    "checkpoint_sha256": sha256_file(best_v2_path)
}

(V2_DRIVE / "metadata.json").write_text(
    json.dumps(metadata_v2, indent=2)
)

# Store processed dataset so it survives Colab reset
dataset_archive = shutil.make_archive(
    "/content/rdd2022_subset_v2",
    "zip",
    root_dir=str(V2_DIR)
)

shutil.copy2(
    dataset_archive,
    DRIVE_ROOT / "datasets/rdd2022_subset_v2.zip"
)

print("V2 model and dataset backed up!")


V2 model and dataset backed up!


In [15]:

import sqlite3
import shutil

BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(str(DB_PATH)) as source:
    with sqlite3.connect(
        str(BACKUP_ROOT / "mlflow.db")
    ) as target:
        source.backup(target)

# Backup artifact directory
backup_artifacts = BACKUP_ROOT / "artifacts"

if backup_artifacts.exists():
    shutil.rmtree(backup_artifacts)

shutil.copytree(ARTIFACT_ROOT, backup_artifacts)

print("MLflow tracking backup complete!")


MLflow tracking backup complete!
